# Capstone pipeline — EgoExo-Fitness + BiLSTM (Google Colab)

This notebook maps your **8 task** rubric to the **Finess-coach-capstone** codebase.

| # | Rubric | In this repo |
|---|--------|--------------|
| 1 | Dataset with **exercise type** + **quality** | [EgoExo-Fitness](https://huggingface.co/datasets/Lymann/EgoExo-Fitness) (`interpretable_action_judgement.json`) |
| 2 | Pose → **17 keypoints** | **MediaPipe** (default); YOLO/RTMPose optional in `requirements.txt` |
| 3 | Preprocess keypoints | `apply_keypoint_preprocessing_pipeline` (normalize, impute, FPS sync) |
| 4 | **Biomechanical angles** | `batch_compute_angles_for_index.py` → `*_biomechanics.npz` |
| 5 | **BiLSTM** classification (30-frame windows) | `train_exercise_bilstm.py` |
| 6 | **Second head** — quality regression + text | Same BiLSTM + `--text-supervision`; annotation-only: `train_exercise_annotation_bilstm.py` |
| 7 | **Accuracy, per-class F1, MAE, R²**; **angles vs coords** ablation | `--eval-test`, `test_classification_metrics.json`; `--feature-mode angles\|coords\|mixed` |
| 8 | **Personalisation** | Outline: freeze trunk, small **per-user adapter** (see last section) |

**Before you run:**
- **Runtime → Change runtime type → GPU** (T4 is enough for BiLSTM; MediaPipe angles are CPU-heavy).
- Accept the **gated** dataset terms on Hugging Face.
- **Colab Secrets:** add `HF_TOKEN` (read). Optional: `GITHUB_TOKEN` for private clone.
- **Large data:** connect **Google Drive** and set `DATA_ROOT` under Drive (Colab disk is small).


In [ ]:
# GPU check (BiLSTM trains faster; angle batch is mostly CPU)
!nvidia-smi -L || true
import torch
print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

## 1) Get code + dependencies

Set `REPO_URL` to **your** GitHub repo, **or** mount Drive and `cd` into the project folder.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/YOUR_USER/Finess-coach-capstone-1.git"  # <-- change
BRANCH = "main"
WORK = Path("/content/fitness-coach")

if not WORK.is_dir():
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, str(WORK)], check=True)
else:
    print("Already cloned:", WORK)

os.chdir(WORK)
print("cwd:", os.getcwd())

In [ ]:
# Alternative: open project from Google Drive (comment out clone cell above)
# from google.colab import drive
# drive.mount("/content/drive")
# import os
# os.chdir("/content/drive/MyDrive/path/to/Finess-coach-capstone-1")
# WORK = Path(os.getcwd())

In [ ]:
%pip install -q -r requirements.txt
%pip install -q -e .

import torch
print("cuda:", torch.cuda.is_available())

## 2) Hugging Face token (gated EgoExo)

Use **Colab Secrets** → `HF_TOKEN`, or paste once in the next cell (avoid saving the notebook with the token).

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab Secrets")
except Exception as e:
    print("Secrets not set; set HF_TOKEN manually:", e)
    # os.environ["HF_TOKEN"] = "hf_..."  # only for quick tests

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Secrets or uncomment manual line"

## 3) Download EgoExo-Fitness

- **Default:** `--annotations-only` (fits Colab disk; enough for **annotation BiLSTM** + index with quality).
- **Full frames** (`frames_open` multi-part archives) need **~hundreds of GB** — use **Google Drive** as `DATA_ROOT` and expect long download.

Dataset hub: [Lymann/EgoExo-Fitness](https://huggingface.co/datasets/Lymann/EgoExo-Fitness).

In [ ]:
from pathlib import Path
from google.colab import drive

# Large downloads: mount Drive and point here
drive.mount("/content/drive")
DATA_ROOT = Path("/content/drive/MyDrive/EgoExo-Fitness")  # change if you want
DATA_ROOT.mkdir(parents=True, exist_ok=True)

RESULTS = Path("/content/drive/MyDrive/fitness_coach_results")
RESULTS.mkdir(parents=True, exist_ok=True)
print("DATA_ROOT:", DATA_ROOT)
print("RESULTS:", RESULTS)

In [ ]:
# Annotations-only (recommended first)
!python download_egoexo_fitness_dataset.py \
  --annotations-only \
  --local-dir "{DATA_ROOT}" \
  --min-free-gb 0

# Full snapshot (uncomment ONLY if you have hundreds of GB on Drive — very slow)
# !python download_egoexo_fitness_dataset.py --local-dir "{DATA_ROOT}" --min-free-gb 0

## 4) Build training index + splits (exercise class + quality)

Uses `interpretable_action_judgement.json` → CSV with `exercise_class`, `quality`, `verification_json`, text fields.

In [ ]:
JUDGE = DATA_ROOT / "raw_annotations" / "interpretable_action_judgement.json"
assert JUDGE.is_file(), f"Missing {JUDGE} — run download step"

!python build_egoexo_fitness_index.py \
  --annotations-json "{JUDGE}" \
  --dataset-root "{DATA_ROOT}" \
  --format interpretable \
  --output "{RESULTS}/egoexo_fitness_index.csv"

!python split_exercise_index.py \
  --input "{RESULTS}/egoexo_fitness_index.csv" \
  --output "{RESULTS}/egoexo_fitness_index_split.csv" \
  --test-ratio 0.15 --val-ratio 0.15 --seed 42

## 5) Track A — Annotation-only BiLSTM (no frames required)

Fulfills **tasks 5–6** on **text + key-point verification sequence** only; good when `frames_open` is not local.

**Task 7 metrics:** `--eval-test` writes `test_classification_metrics.json` (accuracy, macro F1, per-class F1, MAE, R²).

In [ ]:
!python train_exercise_annotation_bilstm.py \
  --index-csv "{RESULTS}/egoexo_fitness_index_split.csv" \
  --standardize \
  --balanced-class-weights \
  --epochs 40 \
  --eval-test \
  --output-dir "{RESULTS}/exercise_annotation_bilstm_egoexo"

In [ ]:
# Show test metrics
import json
from pathlib import Path
p = RESULTS / "exercise_annotation_bilstm_egoexo" / "test_classification_metrics.json"
print(json.dumps(json.loads(p.read_text()), indent=2) if p.is_file() else "Run training with --eval-test first")

## 6) Track B — MediaPipe angles + pose BiLSTM (needs `frames_open` on disk)

**Rubric 2–4:** MediaPipe extracts **17 COCO-style keypoints** per frame; pipeline normalizes / imputes / FPS-syncs then computes **8 joint angles**.

Requires extracted frames under `DATA_ROOT` matching CSV `frame_root` (same layout as HF release). If you only have annotations, **skip** this section.

**Ablations (task 7):**
- `--feature-mode angles` (default) vs `coords` vs `mixed`
- For `coords` / `mixed`, run batch with `--save-keypoints` so `*_keypoints.npz` exists next to biomechanics.

In [ ]:
# Uncomment when frames_open is available under DATA_ROOT
ANGLES_DIR = RESULTS / "egoexo_exercise_angles"
ANGLES_DIR.mkdir(parents=True, exist_ok=True)

# !python batch_compute_angles_for_index.py \
#   --index-csv "{RESULTS}/egoexo_fitness_index_split.csv" \
#   --dataset-root "{DATA_ROOT}" \
#   --output-dir "{ANGLES_DIR}" \
#   --egoexo-view ego_l \
#   --save-keypoints

# Dual-head BiLSTM (classification + quality) + text supervision (on by default)
# !python train_exercise_bilstm.py \
#   --index-csv "{RESULTS}/egoexo_fitness_index_split.csv" \
#   --angles-dir "{ANGLES_DIR}" \
#   --standardize \
#   --epochs 30 \
#   --eval-test \
#   --output-dir "{RESULTS}/exercise_bilstm_egoexo"

# Ablation: raw coordinate features (needs *_keypoints.npz in ANGLES_DIR)
# !python train_exercise_bilstm.py \
#   --index-csv "{RESULTS}/egoexo_fitness_index_split.csv" \
#   --angles-dir "{ANGLES_DIR}" \
#   --keypoints-dir "{ANGLES_DIR}" \
#   --feature-mode coords \
#   --standardize \
#   --eval-test \
#   --output-dir "{RESULTS}/exercise_bilstm_egoexo_coords"

## 7) Pose model comparison (rubric 2)

Your repo includes **MediaPipe** (speed/accuracy tradeoff for real-time coaching). Optional: **Ultralytics YOLO-pose**, **rtmlib** (RTMPose/ViTPose-style ONNX) — see `requirements.txt` and project docs.

For a **formal table**, time **ms/frame** and a **proxy accuracy** (e.g. OKS on a labeled subset) in a small benchmark script — not fully automated in one cell; use the same 17-keypoint COCO layout for fair comparison.

## 8) Personalisation layer (rubric 8) — design

**Goal:** keep the **shared** BiLSTM trunk frozen; after each user session, **fine-tune a small adapter** on that user’s data only.

**Concrete options (implement in PyTorch):**
1. **Linear adapter:** freeze `lstm` + first head; add `nn.Linear(d_head, d_head)` + residual before `fc_cls` / `fc_reg`, train adapter + heads on user clips only.
2. **LoRA-style low-rank** layers on LSTM projections (smaller memory than full fine-tune).
3. Store **per-user** checkpoint: `adapter_user_{id}.pt` + shared `exercise_bilstm_best.pt`.

**Data:** session = new windows from that user’s videos or EgoExo clips labeled as that user if you add a `user_id` column to the index.

This notebook stops at **shared-model** training; adapter training is a short fine-tune loop (low LR, few epochs) you add once you define `user_id` in the CSV.

---
### Checklist before submission

- [ ] EgoExo terms accepted; `HF_TOKEN` not committed.
- [ ] Saved `test_classification_metrics.json` + training logs for **annotation** and/or **pose** tracks.
- [ ] Ablation: export metrics for **angles** vs **coords** (or document why only one ran).
- [ ] Personalisation: 1-page design + optional minimal prototype.
